# トークンごとのサプライザルを色で可視化する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/09_transformers/surprisal.ipynb)

NLP クックブック 09-08 ／ Transformers の基本操作

文のどの部分が言語モデルにとって「意外」だったのかを見たい。誤りの検出や、読みにくさの分析に使える。

In [1]:
import math
import torch
from IPython.display import HTML
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name).eval()

def surprisals(text):
    ids = tokenizer(tokenizer.bos_token + text, add_special_tokens=False, return_tensors="pt")["input_ids"]
    with torch.no_grad():
        logp = torch.log_softmax(model(ids).logits[0, :-1], dim=-1)
    s = -logp[torch.arange(ids.shape[1] - 1), ids[0, 1:]] / math.log(2)   # 各トークンの -log2 P(トークン | 文脈)
    return [(tokenizer.decode(i), v) for i, v in zip(ids[0, 1:].tolist(), s.tolist())]

def show(text):
    spans = []
    for tok, s in surprisals(text):
        alpha = min(s / 20, 1.0)
        spans.append(f"<span title='{s:.1f} bit' style='background: rgba(255, 80, 80, {alpha:.2f}); padding: 2px'>{tok}</span>")
    return "<p style='font-size: 1.2em; line-height: 2'>" + "".join(spans) + "</p>"

texts = ["私は毎朝コーヒーを飲んでから会社に行きます。", "私は毎朝コーヒーを食べてから会社を行きます。", "私は毎朝コーヒーを飲んでから月に行きます。"]
for t in texts:
    print(" ".join(f"{tok}({s:.0f})" for tok, s in surprisals(t)))
HTML("".join(show(t) for t in texts))

私は(16) 毎朝(10) コーヒー(8) を飲んで(4) から(5) 会社に(4) 行きます(12) 。(1)
私は(16) 毎朝(10) コーヒー(8) を食べて(10) から(4) 会社を(9) 行きます(10) 。(1)
私は(16) 毎朝(10) コーヒー(8) を飲んで(4) から(5) 月に(21) 行きます(24) 。(1)


- サプライザル (−log₂ 確率) は、心理言語学では読み時間と相関することが知られている指標。
- 文法誤り (「コーヒーを食べて」「会社を行き」) や意味的に意外な語 (「月に」) で値が跳ね上がる。
- 文頭のトークンは文脈がないので、いつも値が大きい。